In [ ]:
import pandas as pd
import numpy as np
import os
import time

from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, random_split

import warnings
warnings.filterwarnings('ignore')

In [92]:
VIDEO_PATH = r'data/Videos/'
LANDMARK_PATH = r'landmarks/'
PROCESSED_LANDMARK_PATH = r'landmarks_processed/'
MODEL_SAVE_PATH = r'models/saved/'
REPORT_PATH = r'VideoReport_updated.csv'

TARGET_FRAMES = 172
TARGET_FPS = 15

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

In [119]:
BATCH_SIZE  = 32
VAL_SPLIT   = 0.15
NUM_WORKERS = 0 #windows
SEED        = 42

In [ ]:
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

## Skeleton Dataset & Loading


In [122]:
class SkeletonDataset(Dataset):
    
    def __init__(self, root: str) -> None:
        self.samples: list[tuple[str, int]] = []
        self.classes: list[str] = sorted(
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for cls in self.classes:
            cls_dir = os.path.join(root, cls)
            for fname in os.listdir(cls_dir):
                if fname.endswith(".npy"):
                    self.samples.append(
                        (os.path.join(cls_dir, fname), self.class_to_idx[cls])
                    )

        print(f"  Dataset: {len(self.samples)} clips, {len(self.classes)} classes")
        print(f"  Classes: {self.classes}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)#(172, 59, 3)
        arr = arr[:, :, :2] # remove conf 

        vel = np.zeros_like(arr)
        vel[1:] = arr[1:] - arr[:-1]
        arr = np.concatenate([arr, vel], axis=-1)

        #normalization
        pos = arr[:, :, :2]
        mu  = pos.mean(axis=(0, 1), keepdims=True)
        std = pos.std(axis=(0, 1), keepdims=True) + 1e-6
        arr[:, :, :2] = (pos - mu) / std

        seq = arr.reshape(172, -1)

        return torch.from_numpy(seq), torch.tensor(label, dtype=torch.long)

In [123]:
full_dataset = SkeletonDataset(PROCESSED_LANDMARK_PATH)
NUM_CLASSES  = len(full_dataset.classes)

n_val   = max(1, int(len(full_dataset) * VAL_SPLIT))
n_train = len(full_dataset) - n_val
train_ds, val_ds = random_split(full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True)

print(f"\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Number of Classes: {NUM_CLASSES}")

all_labels = torch.tensor([full_dataset.samples[i][1] for i in range(len(full_dataset))])
class_counts  = torch.bincount(all_labels, minlength=NUM_CLASSES).float()
class_weights = (class_counts.sum() / (NUM_CLASSES * class_counts)).clamp(max=10.0)
print(f"Class weights: {class_weights.tolist()}")


  Dataset: 535 clips, 26 classes
  Classes: ['1-Ardhpatka', '10-Katakamukha', '11-Suchi', '12-Chandrakala', '13-Padmakosh', '14-Sarpashirsha', '15-Mrigashirsha', '16-Simhamukha', '17-Kangula', '18-Alapadma', '19-Chatura', '2-Kartarimukh', '20-Bhramara', '21-Hamsasya', '22-Hamsapkshaka', '23-Sandamsha', '24-Mukula', '25-Taamrachuda', '26-Trishula', '3-Mayura', '4-Aardhchandra', '5-Aarala', '6-Shukatundaka', '7-Mushti', '8-Shikara', '9-Kapitha']

Train batches: 14 | Val batches: 3
Number of Classes: 26
Class weights: [0.6858974099159241, 1.1431623697280884, 1.1431623697280884, 1.1431623697280884, 0.709549069404602, 1.4697802066802979, 1.1431623697280884, 1.1431623697280884, 1.2104072570800781, 1.1431623697280884, 1.286057710647583, 0.8573718070983887, 0.8573718070983887, 1.1431623697280884, 0.9353147149085999, 0.8573718070983887, 0.8573718070983887, 1.7147436141967773, 0.8573718070983887, 0.8573718070983887, 0.8573718070983887, 0.8573718070983887, 0.8573718070983887, 1.1431623697280884, 

## TCN Model

In [97]:
class ResidualTCNBlock(nn.Module):

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        dilation: int = 1,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.padding = (kernel_size - 1) * dilation

        self.conv1  = nn.Conv1d(in_channels, out_channels,
                                kernel_size=kernel_size,
                                dilation=dilation, padding=self.padding)
        self.bn1    = nn.BatchNorm1d(out_channels)
        self.relu1  = nn.ReLU(inplace=True)
        self.drop1  = nn.Dropout(p=dropout)

        self.conv2  = nn.Conv1d(out_channels, out_channels,
                                kernel_size=kernel_size,
                                dilation=dilation, padding=self.padding)
        self.bn2    = nn.BatchNorm1d(out_channels)

        self.skip = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )
        self.relu_out = nn.ReLU(inplace=True)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in [self.conv1, self.conv2]:
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        if isinstance(self.skip, nn.Conv1d):
            nn.init.kaiming_normal_(self.skip.weight, mode="fan_out", nonlinearity="relu")

    def _causal_conv(self, conv: nn.Conv1d, x: torch.Tensor) -> torch.Tensor:
        """Conv then trim right-side causal padding to preserve T."""
        return conv(x)[:, :, :x.size(2)]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, C_in, T)
        residual = self.skip(x)                          # (B, C_out, T)

        out = self._causal_conv(self.conv1, x)           # (B, C_out, T)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.drop1(out)

        out = self._causal_conv(self.conv2, out)         # (B, C_out, T)
        out = self.bn2(out)

        out = out + residual                             # residual add
        return self.relu_out(out)                        # (B, C_out, T)


In [98]:
class TCNActionRecognizer(nn.Module):

    def __init__(
        self,
        num_classes: int,
        input_dim:   int   = 236,
        proj_dim:    int   = 128,
        kernel_size: int   = 3,
        dilations:   tuple = (1, 2, 4, 8),
        num_heads:   int   = 4,
        attn_dropout: float = 0.1,
        tcn_dropout:  float = 0.2,
        head_dropout: float = 0.4,
    ) -> None:
        super().__init__()

        # 1. Input projection  (B, T, 236) → (B, T, 128)
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(inplace=True),
        )

        # 2. Four residual TCN blocks  (B, 128, T) → (B, 128, T)
        self.tcn_blocks = nn.ModuleList([
            ResidualTCNBlock(proj_dim, proj_dim,
                             kernel_size=kernel_size,
                             dilation=d, dropout=tcn_dropout)
            for d in dilations
        ])
        # Receptive field ≈ 2 × (kernel-1) × Σ(dilations)
        #                 = 2 × 2 × 15 = 60 frames

        # 3. Temporal self-attention  (B, T, 128) → (B, T, 128)
        self.attn_norm    = nn.LayerNorm(proj_dim)
        self.temporal_attn = nn.MultiheadAttention(
            embed_dim=proj_dim, num_heads=num_heads,
            dropout=attn_dropout, batch_first=True,
        )

        # 4. Global average pool  (B, T, 128) → (B, 128)  [implicit]

        # 5. Classification head  (B, 128) → (B, num_classes)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=head_dropout),
            nn.Linear(64, num_classes),
        )

        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:  x – (B, T=172, F=236)
        Returns: logits – (B, num_classes)
        """
        # ── Input projection ──────────────────────────────────
        x = self.input_proj(x)                       # (B, 172, 128)

        # ── TCN backbone ─────────────────────────────────────
        x = x.transpose(1, 2)                        # (B, 128, 172)
        for block in self.tcn_blocks:
            x = block(x)                             # (B, 128, 172)
        x = x.transpose(1, 2)                        # (B, 172, 128)

        # ── Temporal attention ────────────────────────────────
        residual = x
        x_norm   = self.attn_norm(x)                 # (B, 172, 128)  pre-norm
        attn_out, _ = self.temporal_attn(x_norm, x_norm, x_norm)  # (B, 172, 128)
        x = attn_out + residual                      # (B, 172, 128)

        # ── Global average pool ───────────────────────────────
        x = x.mean(dim=1)                            # (B, 128)

        # ── Classification ────────────────────────────────────
        return self.classifier(x)                    # (B, num_classes)


In [99]:
def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def model_summary(model: nn.Module) -> None:
    total     = count_parameters(model, trainable_only=False)
    trainable = count_parameters(model, trainable_only=True)
    print("=" * 52)
    print(f"  Model          : {model.__class__.__name__}")
    print(f"  Total params   : {total:,}")
    print(f"  Trainable      : {trainable:,}")
    print(f"  Non-trainable  : {total - trainable:,}")
    print("=" * 52)


In [100]:
class EarlyStopping:
    def __init__(
        self,
        patience: int  = 10,
        min_delta: float = 1e-4,
        mode: str = "min",
        verbose: bool = True,
    ) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.verbose    = verbose
        self.counter    = 0
        self.best_score: Optional[float] = None
        self.early_stop = False

    def __call__(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False

        improved = (
            score < self.best_score - self.min_delta if self.mode == "min"
            else score > self.best_score + self.min_delta
        )

        if improved:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"    EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


In [101]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
    device: torch.device,
    grad_clip: float = 1.0,
) -> dict:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.perf_counter()

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)   # (B, 172, 236)
        labels = labels.to(device, non_blocking=True)   # (B,)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(inputs)                      # (B, num_classes)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    elapsed = time.perf_counter() - t0
    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "samples_per_sec": total / elapsed,
    }


In [102]:
@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    """Full validation pass (no gradients)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    return {"loss": total_loss / total, "accuracy": correct / total}


In [103]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    *,
    num_classes: int,
    num_epochs: int = 100,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    grad_clip: float = 1.0,
    label_smoothing: float = 0.1,
    class_weights: Optional[torch.Tensor] = None,
    patience: int = 15,
    checkpoint_path: str = "best_tcn.pt",
    device: Optional[torch.device] = None,
) -> dict:
    """
    End-to-end training with AdamW + CosineAnnealingLR + AMP + early stopping.

    Returns history dict: train_loss, train_acc, val_loss, val_acc.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model_summary(model)
    print(f"  Device: {device}\n")

    # Loss
    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights,
                                    label_smoothing=label_smoothing)

    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=weight_decay, betas=(0.9, 0.999))

    # Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr * 1e-2)

    # AMP scaler (no-op on CPU)
    scaler = GradScaler(enabled=device.type == "cuda")

    # Early stopping
    early_stop = EarlyStopping(patience=patience, mode="max", verbose=True)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    hdr = f"{'Epoch':>6} | {'TrLoss':>8} | {'TrAcc':>7} | {'VaLoss':>8} | {'VaAcc':>7} | {'LR':>9}"
    print(hdr)
    print("-" * len(hdr))

    for epoch in range(1, num_epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer,
                             criterion, scaler, device, grad_clip)
        va = validate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["accuracy"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["accuracy"])

        print(f"{epoch:>6} | {tr['loss']:>8.4f} | {tr['accuracy']:>6.2%} | "
              f"{va['loss']:>8.4f} | {va['accuracy']:>6.2%} | {lr_now:>9.2e}")

        # Checkpoint on improvement
        if va["accuracy"] > best_val_acc:
            best_val_acc = va["accuracy"]
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": best_val_acc,
                "classes": train_loader.dataset.dataset.classes
                           if hasattr(train_loader.dataset, "dataset") else [],
            }, checkpoint_path)

        if early_stop(va["accuracy"]):
            print(f"\n  Early stopping at epoch {epoch}.")
            break

    print(f"\n  Best val accuracy: {best_val_acc:.2%}  →  '{checkpoint_path}'")
    return history


In [104]:
@torch.no_grad()
def predict(
    model: nn.Module,
    inputs: torch.Tensor,
    device: Optional[torch.device] = None,
    return_probs: bool = False
):

    if device is None:
        device = next(model.parameters()).device
    model.eval()

    if inputs.dim() == 2:
        inputs = inputs.unsqueeze(0)       # (1, T, F)

    inputs = inputs.to(device)
    with autocast(device_type=device.type, enabled=device.type == "cuda"):
        logits = model(inputs)             # (B, num_classes)

    if return_probs:
        return F.softmax(logits, dim=-1)
    return logits.argmax(dim=-1)


def load_checkpoint(
    checkpoint_path: str,
    model: nn.Module,
    device: Optional[torch.device] = None
):
    if device is None:
        device = torch.device("cpu")
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded '{checkpoint_path}'  "
          f"(epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']:.2%})")
    return model.to(device)


In [ ]:
model = TCNActionRecognizer(
    num_classes  = NUM_CLASSES,
    input_dim    = 236,
    proj_dim     = 128,
    kernel_size  = 3,
    dilations    = (1, 2, 4, 8),
    num_heads    = 4,
    attn_dropout = 0.1,
    tcn_dropout  = 0.2,
    head_dropout = 0.4,
)

# Quick shape check before training
_dummy = torch.randn(2, 172, 236)
assert model(_dummy).shape == (2, NUM_CLASSES), "Shape mismatch!"
print(f"Forward-pass shape check passed: (2, 172, 236) → (2, {NUM_CLASSES})")

history = train(
    model,
    train_loader,
    val_loader,
    num_classes     = NUM_CLASSES,
    num_epochs      = 100,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    grad_clip       = 1.0,
    label_smoothing = 0.1,
    class_weights   = class_weights,   # computed in cell 4
    patience        = 15,
    checkpoint_path = os.path.join(MODEL_SAVE_PATH, "best_tcn.pt"),
    device          = DEVICE,
)

Forward-pass shape check passed: (2, 172, 236) → (2, 26)
  Model          : TCNActionRecognizer
  Total params   : 503,130
  Trainable      : 503,130
  Non-trainable  : 0
  Device: cuda

 Epoch |   TrLoss |   TrAcc |   VaLoss |   VaAcc |        LR
------------------------------------------------------------
     1 |   3.2711 |  6.25% |   3.2745 | 11.25% |  3.00e-04
     2 |   3.2443 |  9.94% |   3.2564 |  6.88% |  3.00e-04
    EarlyStopping counter: 1/15
     3 |   3.1972 | 13.35% |   3.2168 |  8.12% |  2.99e-04
    EarlyStopping counter: 2/15
     4 |   3.1217 | 15.06% |   3.1610 |  8.75% |  2.99e-04
    EarlyStopping counter: 3/15
     5 |   3.0662 | 15.06% |   3.1017 | 18.12% |  2.98e-04
     6 |   2.9881 | 15.62% |   3.0232 | 15.00% |  2.97e-04
    EarlyStopping counter: 1/15
     7 |   2.9032 | 20.74% |   2.9302 | 19.38% |  2.96e-04
     8 |   2.7832 | 21.88% |   2.8669 | 17.50% |  2.95e-04
    EarlyStopping counter: 1/15
     9 |   2.6588 | 20.45% |   2.6921 | 15.62% |  2.94e-04


## BiLSTM Model

In [153]:
class BiLSTMActionRecognizer(nn.Module):
    def __init__(
        self,
        num_classes:  int,
        input_dim:    int = 236,
        proj_dim:     int = 128,
        hidden_dim:   int = 256,
        num_layers:   int = 2,
        num_heads:    int = 4,
        attn_dropout: float = 0.1,
        lstm_dropout: float = 0.3,
        head_dropout: float = 0.4,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.ReLU(),
        )

        self.bilstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
        )

        self.proj = nn.Linear(hidden_dim * 2, proj_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=proj_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.ReLU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x, _ = self.bilstm(x)
        x = self.proj(x)

        attn_out, _ = self.attn(x, x, x)
        x = x + attn_out

        x = x.mean(dim=1)
        return self.classifier(x)

In [154]:
def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def model_summary(model: nn.Module) -> None:
    total     = count_parameters(model, trainable_only=False)
    trainable = count_parameters(model, trainable_only=True)
    print("=" * 52)
    print(f"  Model          : {model.__class__.__name__}")
    print(f"  Total params   : {total:,}")
    print(f"  Trainable      : {trainable:,}")
    print(f"  Non-trainable  : {total - trainable:,}")
    print("=" * 52)

In [155]:
class EarlyStopping:
    def __init__(
        self,
        patience:  int   = 10,
        min_delta: float = 1e-4,
        mode:      str   = "min",
        verbose:   bool  = True,
    ) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.verbose    = verbose
        self.counter    = 0
        self.best_score: Optional[float] = None
        self.early_stop = False

    def __call__(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False

        improved = (
            score < self.best_score - self.min_delta if self.mode == "min"
            else score > self.best_score + self.min_delta
        )

        if improved:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"    EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

In [156]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler:    GradScaler,
    device:    torch.device,
    grad_clip: float = 1.0,
) -> dict:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.perf_counter()

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    elapsed = time.perf_counter() - t0
    return {
        "loss":            total_loss / total,
        "accuracy":        correct / total,
        "samples_per_sec": total / elapsed,
    }

In [157]:
@torch.no_grad()
def validate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> dict:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * inputs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    return {
        "loss":     total_loss / total,
        "accuracy": correct / total,
    }

In [158]:
def train(
    model:           nn.Module,
    train_loader:    DataLoader,
    val_loader:      DataLoader,
    *,
    num_classes:     int,
    num_epochs:      int   = 100,
    lr:              float = 3e-4,
    weight_decay:    float = 1e-4,
    grad_clip:       float = 1.0,
    label_smoothing: float = 0.1,
    class_weights:   Optional[torch.Tensor] = None,
    patience:        int   = 15,
    checkpoint_path: str   = "best_bilstm.pt",
    device:          Optional[torch.device] = None,
) -> dict:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model_summary(model)
    print(f"  Device: {device}\n")

    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=label_smoothing,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr * 1e-2,
    )

    scaler     = GradScaler(device="cuda" if device.type == "cuda" else "cpu")
    early_stop = EarlyStopping(patience=patience, mode="max", verbose=True)

    history    = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    hdr = f"{'Epoch':>6} | {'TrLoss':>8} | {'TrAcc':>7} | {'VaLoss':>8} | {'VaAcc':>7} | {'LR':>9}"
    print(hdr)
    print("-" * len(hdr))

    for epoch in range(1, num_epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer,
                             criterion, scaler, device, grad_clip)
        va = validate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["accuracy"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["accuracy"])

        print(f"{epoch:>6} | {tr['loss']:>8.4f} | {tr['accuracy']:>6.2%} | "
              f"{va['loss']:>8.4f} | {va['accuracy']:>6.2%} | {lr_now:>9.2e}")

        if va["accuracy"] > best_val_acc:
            best_val_acc = va["accuracy"]
            torch.save({
                "epoch":              epoch,
                "model_state_dict":   model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc":            best_val_acc,
                "classes":            train_loader.dataset.dataset.classes
                                      if hasattr(train_loader.dataset, "dataset") else [],
            }, checkpoint_path)

        if early_stop(va["accuracy"]):
            print(f"\n  Early stopping at epoch {epoch}.")
            break

    print(f"\n  Best val accuracy: {best_val_acc:.2%}  →  '{checkpoint_path}'")
    return history

In [159]:
@torch.no_grad()
def predict_from_path_bilstm(
    npy_path:        str,
    checkpoint_path: str,
    classes:         list[str],
    num_classes:     Optional[int] = None,
    device:          Optional[torch.device] = None,
) -> dict:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if num_classes is None:
        num_classes = len(classes)

    ckpt  = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = BiLSTMActionRecognizer(
        num_classes=num_classes,
        input_dim=236,
        proj_dim=128,
        hidden_dim=128,
        num_layers=2,
        num_heads=4,
        attn_dropout=0.1,
        lstm_dropout=0.2,
        head_dropout=0.4,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()

    arr = np.load(npy_path).astype(np.float32)          # (172, 59, 3)
    arr = arr[:, :, :2]                                 # (172, 59, 2)
    vel = np.zeros_like(arr)
    vel[1:] = arr[1:] - arr[:-1]
    arr = np.concatenate([arr, vel], axis=-1)           # (172, 59, 4)
    pos = arr[:, :, :2]
    mu  = pos.mean(axis=(0, 1), keepdims=True)
    std = pos.std(axis=(0, 1), keepdims=True) + 1e-6
    arr[:, :, :2] = (pos - mu) / std
    seq    = arr.reshape(172, -1)                       # (172, 236)
    tensor = torch.from_numpy(seq).unsqueeze(0).to(device)

    with autocast(device_type=device.type):
        logits = model(tensor)

    probs = F.softmax(logits, dim=-1).squeeze(0)
    top3_vals, top3_idxs = probs.topk(min(3, num_classes))
    top3 = [
        {"class": classes[i.item()], "idx": i.item(), "confidence": round(v.item(), 4)}
        for v, i in zip(top3_vals, top3_idxs)
    ]

    return {
        "predicted_class": top3[0]["class"],
        "predicted_idx":   top3[0]["idx"],
        "confidence":      top3[0]["confidence"],
        "top3":            top3,
        "all_probs":       {classes[i]: round(probs[i].item(), 4) for i in range(num_classes)},
    }

In [ ]:

model_bilstm = BiLSTMActionRecognizer(
    num_classes  = NUM_CLASSES,
    input_dim    = 236,
    proj_dim     = 128,
    hidden_dim   = 512,
    num_layers   = 4,
    num_heads    = 4,
    attn_dropout = 0.1,
    lstm_dropout = 0.2,
    head_dropout = 0.3,
)

_dummy = torch.randn(2, 172, 236)
assert model_bilstm(_dummy).shape == (2, NUM_CLASSES), "Shape mismatch!"
print(f"Forward-pass check passed: (2, 172, 236) → (2, {NUM_CLASSES})")

history_bilstm = train(
    model_bilstm,
    train_loader,
    val_loader,
    num_classes     = NUM_CLASSES,
    num_epochs      = 100,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    grad_clip       = 0.5,
    label_smoothing = 0.1,
    class_weights   = class_weights,
    patience        = 15,
    checkpoint_path = "models/saved/best_bilstm.pt",
    device          = DEVICE,
)

Forward-pass check passed: (2, 172, 236) → (2, 26)
  Model          : BiLSTMActionRecognizer
  Total params   : 21,776,026
  Trainable      : 21,776,026
  Non-trainable  : 0
  Device: cuda

 Epoch |   TrLoss |   TrAcc |   VaLoss |   VaAcc |        LR
------------------------------------------------------------
     1 |   3.2779 |  4.24% |   3.2842 |  6.25% |  3.00e-04
     2 |   3.2758 |  3.57% |   3.2872 |  6.25% |  3.00e-04
    EarlyStopping counter: 1/15
     3 |   3.2733 |  4.69% |   3.2922 |  1.25% |  2.99e-04
    EarlyStopping counter: 2/15
     4 |   3.2739 |  3.35% |   3.2911 |  1.25% |  2.99e-04
    EarlyStopping counter: 3/15
     5 |   3.2694 |  6.03% |   3.3122 |  1.25% |  2.98e-04
    EarlyStopping counter: 4/15
     6 |   3.2720 |  5.58% |   3.2742 |  1.25% |  2.97e-04
    EarlyStopping counter: 5/15
     7 |   3.2110 |  6.03% |   3.2157 |  3.75% |  2.96e-04
    EarlyStopping counter: 6/15
     8 |   3.1193 |  7.37% |   3.2586 |  3.75% |  2.95e-04
    EarlyStopping counte

## Evaluation

In [117]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    classes: list[str],
    device: Optional[torch.device] = None,
) -> dict:
    """
    Full evaluation pass with per-class metrics.

    Args:
        model   : Trained TCNActionRecognizer.
        loader  : DataLoader for the evaluation set.
        classes : Ordered list of class names.
        device  : Target device. Auto-detected if None.

    Returns:
        dict with keys:
            accuracy        (float)       – overall accuracy
            avg_loss        (float)       – average cross-entropy loss
            class_report    (dict)        – per-class precision, recall, f1, support
            confusion_matrix (np.ndarray) – (num_classes, num_classes)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_classes = len(classes)
    criterion   = nn.CrossEntropyLoss()

    model.to(device).eval()

    all_preds  = []
    all_labels = []
    total_loss = 0.0
    total      = 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(inputs)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * inputs.size(0)
        total      += inputs.size(0)

        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    # ── Overall accuracy ───────────────────────────────────────────────────
    accuracy = (all_preds == all_labels).float().mean().item()

    # ── Confusion matrix ───────────────────────────────────────────────────
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    for true, pred in zip(all_labels.tolist(), all_preds.tolist()):
        confusion[true][pred] += 1

    # ── Per-class precision, recall, F1 ───────────────────────────────────
    class_report = {}
    for i, cls in enumerate(classes):
        tp = confusion[i, i]
        fp = confusion[:, i].sum() - tp
        fn = confusion[i, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)

        class_report[cls] = {
            "precision": round(precision, 4),
            "recall":    round(recall,    4),
            "f1":        round(f1,        4),
            "support":   int(confusion[i, :].sum()),
        }

    # ── Print summary ──────────────────────────────────────────────────────
    print(f"\n{'='*58}")
    print(f"  Overall accuracy : {accuracy:.2%}")
    print(f"  Average loss     : {total_loss / total:.4f}")
    print(f"{'='*58}")
    print(f"  {'Class':<20} {'Prec':>6} {'Rec':>6} {'F1':>6} {'N':>5}")
    print(f"  {'-'*47}")
    for cls, m in class_report.items():
        print(f"  {cls:<20} {m['precision']:>6.2%} {m['recall']:>6.2%} "
              f"{m['f1']:>6.2%} {m['support']:>5}")
    print(f"{'='*58}\n")

    return {
        "accuracy":         accuracy,
        "avg_loss":         round(total_loss / total, 4),
        "class_report":     class_report,
        "confusion_matrix": confusion,
    }

In [ ]:
# After training, load best checkpoint and evaluate on val set
model = ("models/saved/best_bilstm.pt", model, device=DEVICE)

results = evaluate(model, val_loader, classes=full_dataset.classes, device=DEVICE)


cm_df = pd.DataFrame(
    results["confusion_matrix"],
    index=full_dataset.classes,
    columns=full_dataset.classes,
)
print(cm_df)

RuntimeError: Error(s) in loading state_dict for TCNActionRecognizer:
	Missing key(s) in state_dict: "tcn_blocks.0.conv1.weight", "tcn_blocks.0.conv1.bias", "tcn_blocks.0.bn1.weight", "tcn_blocks.0.bn1.bias", "tcn_blocks.0.bn1.running_mean", "tcn_blocks.0.bn1.running_var", "tcn_blocks.0.conv2.weight", "tcn_blocks.0.conv2.bias", "tcn_blocks.0.bn2.weight", "tcn_blocks.0.bn2.bias", "tcn_blocks.0.bn2.running_mean", "tcn_blocks.0.bn2.running_var", "tcn_blocks.1.conv1.weight", "tcn_blocks.1.conv1.bias", "tcn_blocks.1.bn1.weight", "tcn_blocks.1.bn1.bias", "tcn_blocks.1.bn1.running_mean", "tcn_blocks.1.bn1.running_var", "tcn_blocks.1.conv2.weight", "tcn_blocks.1.conv2.bias", "tcn_blocks.1.bn2.weight", "tcn_blocks.1.bn2.bias", "tcn_blocks.1.bn2.running_mean", "tcn_blocks.1.bn2.running_var", "tcn_blocks.2.conv1.weight", "tcn_blocks.2.conv1.bias", "tcn_blocks.2.bn1.weight", "tcn_blocks.2.bn1.bias", "tcn_blocks.2.bn1.running_mean", "tcn_blocks.2.bn1.running_var", "tcn_blocks.2.conv2.weight", "tcn_

In [ ]:
print(f"Total clips   : {len(full_dataset)}")
print(f"Train clips   : {len(train_ds)}")
print(f"Val clips     : {len(val_ds)}")
print(f"Clips/class   : {len(full_dataset) // NUM_CLASSES} avg")

# Per-class count
from collections import Counter
label_counts = Counter(full_dataset.samples[i][1] for i in range(len(full_dataset)))
for cls, idx in full_dataset.class_to_idx.items():
    print(f"  {cls:<25} {label_counts[idx]:>4} clips")

Total clips   : 535
Train clips   : 375
Val clips     : 160
Clips/class   : 20 avg
  1-Ardhpatka                 30 clips
  10-Katakamukha              18 clips
  11-Suchi                    18 clips
  12-Chandrakala              18 clips
  13-Padmakosh                29 clips
  14-Sarpashirsha             14 clips
  15-Mrigashirsha             18 clips
  16-Simhamukha               18 clips
  17-Kangula                  17 clips
  18-Alapadma                 18 clips
  19-Chatura                  16 clips
  2-Kartarimukh               24 clips
  20-Bhramara                 24 clips
  21-Hamsasya                 18 clips
  22-Hamsapkshaka             22 clips
  23-Sandamsha                24 clips
  24-Mukula                   24 clips
  25-Taamrachuda              12 clips
  26-Trishula                 24 clips
  3-Mayura                    24 clips
  4-Aardhchandra              24 clips
  5-Aarala                    24 clips
  6-Shukatundaka              24 clips
  7-Mushti          